# 1. Import Libraries

In [ ]:
from ollama import chat
from pathlib import Path
import pandas as pd
import numpy as np
import random
import json
import csv
import re

OUTPUT_FILE_NAME = "interactions.csv"
CARDS_PATH = Path.cwd().parent.joinpath("Data/all_ygo_cards.csv")

# 2. Import Data

In [ ]:
MODEL_NAME = "llama3.1:8b"
RDMNESS = -1
with open("prompt.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()
dfCards = pd.read_csv(CARDS_PATH)

display(dfCards)

,id,name,desc,atk,defe,attribute,type,frametype,level,card_sets
0,80181649,"""A Case for K9""","When this card is activated: You can add 1 ""K9...",NaN,NaN,NaN,Continuous,spell,NaN,Justice Hunters | Justice Hunters
1,34541863,"""A"" Cell Breeding Device","During each of your Standby Phases, put 1 A-Co...",NaN,NaN,NaN,Continuous,spell,NaN,Force of the Breaker
2,64163367,"""A"" Cell Incubator",Each time an A-Counter(s) is removed from play...,NaN,NaN,NaN,Continuous,spell,NaN,Gladiator's Assault
3,91231901,"""A"" Cell Recombination Device",Target 1 face-up monster on the field; send 1 ...,NaN,NaN,NaN,Quick-Play,spell,NaN,Invasion: Vengeance
4,73262676,"""A"" Cell Scatter Burst","Select 1 face-up ""Alien"" monster you control. ...",NaN,NaN,NaN,Quick-Play,spell,NaN,Strike of Neos
...,...,...,...,...,...,...,...,...,...,...
14132,2648201,ZW - Sleipnir Mail,"You can target 1 ""Utopia"" monster you control;...",1000.0,1000.0,LIGHT,Beast,effect,4.0,Primal Origin
14133,95886782,ZW - Sylphid Wing,"You can only control 1 ""ZW - Sylphid Wing"". Yo...",800.0,1600.0,LIGHT,Beast,effect,4.0,Brothers of Legend
14134,81471108,ZW - Tornado Bringer,"You can target 1 ""Utopia"" monster you control;...",1300.0,1800.0,WIND,Dragon,effect,5.0,Cosmo Blazer | King's Court
14135,18865703,ZW - Ultimate Shield,When this card is Normal or Special Summoned: ...,0.0,2000.0,EARTH,Aqua,effect,4.0,Cosmo Blazer | King's Court


# 2. LLM and Prompt Functions 

In [3]:
cards_by_id = dfCards.set_index("id")

def get_card(card_id):
    row = cards_by_id.loc[card_id]
    return row.to_dict()


def build_user_prompt(idA, idB):
    cardA = cards_by_id.loc[idA]
    cardB = cards_by_id.loc[idB]
    
    return f"""
CARD_A
id: {idA}
name: {cardA['name']}
desc: {cardA['desc']}
atk: {cardA['atk']}
defe: {cardA['defe']}
attribute: {cardA['attribute']}
type: {cardA['type']}
frametype: {cardA['frametype']}
level: {cardA['level']}

CARD_B
id: {idB}
name: {cardB['name']}
desc: {cardB['desc']}
atk: {cardB['atk']}
defe: {cardB['defe']}
attribute: {cardB['attribute']}
type: {cardB['type']}
frametype: {cardB['frametype']}
level: {cardB['level']}
"""

In [4]:
def ask_llm(user_prompt: str, model: str = MODEL_NAME, sys_prompt: str = SYSTEM_PROMPT) -> str:
    response = chat(
        model=model,
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_prompt},
        ],
		options={"temperature": 0.2,
        	"num_predict": 1600,
            "top_p": 0.8,
            "top_k": 100,
            "num_ctx": 16384}
    )
    return response["message"]["content"]

# 3. Draw Random Cards

In [5]:
cardset_list = {j for i in dfCards['card_sets'].dropna() for j in i.split(" | ")}

def randomCards():
    pair = dfCards.sample(2)
    cardA = pair.iloc[0]
    cardB = pair.iloc[1]
    return cardA['id'], cardB['id']

def randomCardsSameSet():
    set_chosen = random.choice(list(cardset_list))
    cards_in_set = dfCards[dfCards['card_sets'].str.contains(set_chosen, na=False)] 
    #print(set_chosen)
    if len(cards_in_set) < 2:
        return randomCards()
    pair = cards_in_set.sample(2)
    cardA = pair.iloc[0]
    cardB = pair.iloc[1]
    return cardA['id'], cardB['id']

def drawCards(Randomness = RDMNESS):
	if random.random() < Randomness:
		return randomCards()
	else:
		return randomCardsSameSet()
          

# 4. Automated Data Annotation

In [6]:
def annotatePair():
    idA, idB = drawCards()
    prompt = build_user_prompt(idA, idB)
    result = ask_llm(prompt)
    return idA, idB, result

In [7]:
CSV_COLUMNS = ['id_cardA', 'id_cardB',
'destroy','references_type','mill_from_deck',
'mentions_name','equip','mentions_archetype',
'link_summon','reduce_atk','negate_activation',
'pendulum_summon','revive_from_graveyard','protect_monster',
'discard','gain_lp','change_attribute',
'return_from_graveyard','add_to_hand','target',
'normal_summon','references_archetype','force_attack',
'send_to_graveyard','piercing_damage','return_to_deck',
'excavate_deck','search_deck','change_type',
'change_control','xyz_summon','take_control',
'unequip','negate_effect','cannot_be_targeted',
'direct_attack','tribute_summon','cannot_attack',
'pay_lp','references_def','references_frametype',
'special_summon','fusion_summon','return_to_hand',
'protect_effects','draw_cards','references_attribute',
'banish','references_atk','set_from_deck','change_level',
'buff_def','skip_phase','references_level','reduce_def',
'cannot_be_destroyed','buff_atk','synchro_summon']

RELATION_COLUMNS = CSV_COLUMNS[2:]


In [8]:
def extract_json_block(t):
    match = re.search(r'\{[\s\S]*\}', t)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

def annotatePair_row():
    
	idA, idB, resp = annotatePair()
	row = {col: 0 for col in CSV_COLUMNS}
	row["id_cardA"] = idA
	row["id_cardB"] = idB

	data = extract_json_block(resp)
	if data is None:
		return row

	relations = data.get("relations", [])
	for rel in relations:
		rel_type = rel.get("relation_type")
		if rel_type in RELATION_COLUMNS:
			row[rel_type] = 1

	return row

def add_rows_to_csv(N, output=OUTPUT_FILE_NAME):
    with open(output, "a", newline="", encoding="utf-8") as f:
        for _ in range(N):
            writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
            writer.writerow(annotatePair_row())      

In [9]:
add_rows_to_csv(10000, output=OUTPUT_FILE_NAME)

C:\Users\fares\AppData\Local\Temp\ipykernel_11928\3130196337.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  cards_in_set = dfCards[dfCards['card_sets'].str.contains(set_chosen, na=False)]


In [10]:
df = pd.read_csv(OUTPUT_FILE_NAME)
display(df)

,id_cardA,id_cardB,destroy,references_type,mill_from_deck,mentions_name,equip,mentions_archetype,link_summon,reduce_atk,...,references_atk,set_from_deck,change_level,buff_def,skip_phase,references_level,reduce_def,cannot_be_destroyed,buff_atk,synchro_summon
0,47297616,78792195,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,57285770,49154689,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,26902560,11471117,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,80316585,12923641,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,55343172,40975574,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27412,96795312,77864539,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
27413,81275309,43332022,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
27414,84766279,27944249,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
27415,30213599,34487429,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
